## Baselines

In [1]:
import os
from estnltk.converters import json_to_text

Ajafaktide korpusel masinõppeeksperimentide järgselt baasmudeli ehk sagedaseima klassi ennustamise õigsuse skooride leidmine kestustüüpide määramise ja sündmuse-ajaväljendi ajaseoste määramise eksperimentide tulemustega kõrvutamiseks.

Ajafaktide korpuses oli:
 - *train_larger* treenimiseks (uudised)
 - *test* testimiseks (uudised)
 - *rkogu* täiendavaks testimiseks (riigikogu stenogrammid)
 - *horisont* täiendavaks testimiseks (ajalooalased teadusartiklid)

In [2]:
temp_fact_news_path = 'model_data/TempFact/test/'
temp_fact_rkogu_path = 'model_data/TempFact/rkogu/'
temp_fact_horisont_path = 'model_data/TempFact/horisont/'

In [3]:
news_texts = []

for filename in os.listdir(temp_fact_news_path):
    text_obj = json_to_text(file=temp_fact_news_path + filename)
    news_texts.append(text_obj)
    
rkogu_texts = []

for filename in os.listdir(temp_fact_rkogu_path):
    text_obj = json_to_text(file=temp_fact_rkogu_path + filename)
    rkogu_texts.append(text_obj)
    
horisont_texts = []

for filename in os.listdir(temp_fact_horisont_path):
    text_obj = json_to_text(file=temp_fact_horisont_path + filename)
    horisont_texts.append(text_obj)

In [4]:
news_texts[0]

Text(text='Kuidagi ei saa unustada eestlaste poliitilist aktiivsust nõukogude võimu lõpuaastatel .\nSiis oli kogu rahval kindel soov ja tahtmine tagasi saada oma endist Eestimaad , kus olid tagatud õigused oma varale ja kindlustatud tulevik lastele .\nVahel tekib tunne , et võimu juurde on saanud valed inimesed .\nEi ole suudetud kehtetuks tunnistada isegi nõukogudeaegset ülekohut ja varade riisumist .\nOn hakatud kasutama venitamise taktikat ja sellega jämedalt rikutud demokraatia reegleid ja rahvusvaheliselt tunnustatud omandiõigusi .\nPraegust elu jälgides võib tõdeda , et meie majandusreformid on ebaõnnestunud ja praegune üliliberaalne majandussüsteem on takistanud meie majanduse normaalset arengut , sest meie turg on olnud reguleerimata .\nRiik saaks olla ainult siis rikas , kui tal on tugev majandus .\nEluga tuleb edasi liikuda , nõustuda meie maasaadikute ettepanekuga ja jätta maha praegune tollivaba majandus .\nEestile oleks kiiresti vaja korralikku hinna- ja maksupoliitikat .\nNüüd , mil õnnelikult on lõpule viidud riigimeeste järjekordne palkade korrastamine , oleks aeg mõtlema hakata , et ka põllumeeste töö saaks väärtustatud .\n')

In [5]:
news_texts[0].events

Layer(name='events', attributes=('brat_id', 'class', 'class_confidence', 'duration', 'duration_confidence', 'comment'), spans=SL[EnvelopingSpan(['aktiivsust'], [{'brat_id': 'T6', 'class': 'STATE', 'class_confidence': 'high', 'duration': 'years', 'duration_confidence': 'high', 'comment': None}]),
EnvelopingSpan(['elu'], [{'brat_id': 'T9', 'class': 'STATE', 'class_confidence': 'high', 'duration': 'years', 'duration_confidence': 'high', 'comment': None}]),
EnvelopingSpan(['majandussüsteem'], [{'brat_id': 'T7', 'class': 'STATE', 'class_confidence': 'high', 'duration': 'years', 'duration_confidence': 'neutral', 'comment': None}]),
EnvelopingSpan(['majandus'], [{'brat_id': 'T8', 'class': 'STATE', 'class_confidence': 'high', 'duration': 'years', 'duration_confidence': 'neutral', 'comment': None}]),
EnvelopingSpan(['lõpule', 'viidud'], [{'brat_id': 'T10', 'class': 'ASPECTUAL', 'class_confidence': 'high', 'duration': 'hours', 'duration_confidence': 'low', 'comment': None}]),
EnvelopingSpan(['korrastamine'], [{'brat_id': 'T11', 'class': 'I_ACTION', 'class_confidence': 'high', 'duration': 'weeks', 'duration_confidence': 'neutral', 'comment': None}])])

### Durations

In [6]:
def get_event_labels(text_list):
    """
    Leiab ja tagastab sündmuste ajaliste kestuste labelid juhul, kui sündmusfraasile leidub vastav peasõna.
    Sest treenimisel-testimisel jäeti välja sündmusfraasid, millele peaõnade kihil vastet ei leidunud.
    """
    duration_labels = []
    n_event_phrases_total = 0
    for text_idx, text in enumerate(text_list):
        n_event_phrases_total+=len(text.events)
        # sündmusfraasi peasõna
        for idx, word in enumerate(text.gold_word_events_main):
            if word.nertag == 'B-EVENT' or word.nertag == 'I-EVENT':
                #has_corresponding_phrase = False
                word_span = text.words.get(word[0])
                event_phrase = None
                duration_label = None
                for idx2, event in enumerate(text.events):
                    for event_word in event:
                        if text.words.get(event_word) == word_span:
                            event_phrase = event
                            duration_label = event.duration
                            break
                    if event_phrase is not None and duration_label is not None:
                        break
            
                if word_span and event_phrase is not None and duration_label is not None:
                    duration_labels.append(duration_label)
                
    return duration_labels, n_event_phrases_total

In [7]:
news_labels, n_news_event_phrases = get_event_labels(news_texts)
rkogu_labels, n_rkogu_event_phrases = get_event_labels(rkogu_texts)
horisont_labels, n_horisont_event_phrases = get_event_labels(horisont_texts)

In [8]:
print(f"Uudistes labeleid {len(news_labels)}, sündmusfraase kokku {n_news_event_phrases}")
print(f"Stenogrammides labeleid {len(rkogu_labels)}, sündmusfraase kokku {n_rkogu_event_phrases}")
print(f"Ajalooalastes teadusartiklites labeleid {len(horisont_labels)}, sündmusfraase kokku {n_horisont_event_phrases}")

Uudistes labeleid 90, sündmusfraase kokku 94
Stenogrammides labeleid 100, sündmusfraase kokku 107
Ajalooalastes teadusartiklites labeleid 90, sündmusfraase kokku 94


In [12]:
from collections import Counter

def calculate_baseline_accuracy_score(label_list):
    most_frequent = Counter(label_list).most_common(1)[0]
    return most_frequent[0], most_frequent[1] / len(label_list)

def calculate_baseline_accuracy_score_agg(label_list):
    agg_dct = {"instant": "less_than_day",
               "seconds": "less_than_day",
               "minutes": "less_than_day",
               "hours": "less_than_day",
               "days": "less_than_year",
               "weeks": "less_than_year",
               "months": "less_than_year",
               "years": "more_than_year",
               "centuries": "more_than_year",
               "forever": "more_than_year"}
    
    label_list = [agg_dct[l] for l in label_list]
    most_frequent = Counter(label_list).most_common(1)[0]
    return most_frequent[0], most_frequent[1] / len(label_list)

def calculate_baseline_accuracy_score_agg_binary(label_list):
    agg_dct = {"instant": "less_than_day",
               "seconds": "less_than_day",
               "minutes": "less_than_day",
               "hours": "less_than_day",
               "days": "more_than_day",
               "weeks": "more_than_day",
               "months": "more_than_day",
               "years": "more_than_day",
               "centuries": "more_than_day",
               "forever": "more_than_day"}
    
    label_list = [agg_dct[l] for l in label_list]
    most_frequent = Counter(label_list).most_common(1)[0]
    return most_frequent[0], most_frequent[1] / len(label_list)

In [13]:
news_most_freq_label, news_baseline_accuracy = calculate_baseline_accuracy_score(news_labels)
rkogu_most_freq_label, rkogu_baseline_accuracy = calculate_baseline_accuracy_score(rkogu_labels)
horisont_most_freq_label, horisont_baseline_accuracy = calculate_baseline_accuracy_score(horisont_labels)

news_most_freq_label_agg, news_baseline_accuracy_agg = calculate_baseline_accuracy_score_agg(news_labels)
rkogu_most_freq_label_agg, rkogu_baseline_accuracy_agg = calculate_baseline_accuracy_score_agg(rkogu_labels)
horisont_most_freq_label_agg, horisont_baseline_accuracy_agg = calculate_baseline_accuracy_score_agg(horisont_labels)

news_most_freq_label_agg_binary, news_baseline_accuracy_agg_binary = calculate_baseline_accuracy_score_agg_binary(news_labels)
rkogu_most_freq_label_agg_binary, rkogu_baseline_accuracy_agg_binary = calculate_baseline_accuracy_score_agg_binary(rkogu_labels)
horisont_most_freq_label_agg_binary, horisont_baseline_accuracy_agg_binary = calculate_baseline_accuracy_score_agg_binary(horisont_labels)



In [11]:
print(f"Uudistes sagedaseim klass: {news_most_freq_label}, baseline_accuracy: {news_baseline_accuracy}")
print(f"Stenogrammides sagedaseim klass: {rkogu_most_freq_label}, baseline_accuracy: {rkogu_baseline_accuracy}")
print(f"Ajaloos sagedaseim klass: {horisont_most_freq_label}, baseline_accuracy: {horisont_baseline_accuracy}")

Uudistes sagedaseim klass: minutes, baseline_accuracy: 0.3
Stenogrammides sagedaseim klass: hours, baseline_accuracy: 0.19
Ajaloos sagedaseim klass: centuries, baseline_accuracy: 0.35555555555555557


In [14]:
print(f"Uudistes sagedaseim agreg. klass: {news_most_freq_label_agg}, baseline_accuracy: {news_baseline_accuracy_agg}")
print(f"Stenogrammides sagedaseim agreg. klass: {rkogu_most_freq_label_agg}, baseline_accuracy: {rkogu_baseline_accuracy_agg}")
print(f"Ajaloos sagedaseim agreg. klass: {horisont_most_freq_label_agg}, baseline_accuracy: {horisont_baseline_accuracy_agg}")

Uudistes sagedaseim agreg. klass: less_than_day, baseline_accuracy: 0.43333333333333335
Stenogrammides sagedaseim agreg. klass: less_than_day, baseline_accuracy: 0.62
Ajaloos sagedaseim agreg. klass: more_than_year, baseline_accuracy: 0.5111111111111111


In [15]:
print(f"Uudistes sagedaseim binaarne agreg. klass: {news_most_freq_label_agg_binary}, baseline_accuracy: {news_baseline_accuracy_agg_binary}")
print(f"Stenogrammides sagedaseim binaarne agreg. klass: {rkogu_most_freq_label_agg_binary}, baseline_accuracy: {rkogu_baseline_accuracy_agg_binary}")
print(f"Ajaloos sagedaseim binaarne agreg. klass: {horisont_most_freq_label_agg_binary}, baseline_accuracy: {horisont_baseline_accuracy_agg_binary}")

Uudistes sagedaseim binaarne agreg. klass: more_than_day, baseline_accuracy: 0.5666666666666667
Stenogrammides sagedaseim binaarne agreg. klass: less_than_day, baseline_accuracy: 0.62
Ajaloos sagedaseim binaarne agreg. klass: more_than_day, baseline_accuracy: 0.7888888888888889


### Event-timex TLINKS

In [22]:
# finds timex phrases of given sentence, if they exist in tlink layer
def get_sentence_phrases(text_obj, sentence, tlink_layer_name):
    phrase_matches = []
    for i in range(len(text_obj[tlink_layer_name])):
        phrase = []
        cur_tlink = text_obj[tlink_layer_name][i]
        for j in range(len(cur_tlink.b_text.base_span)):
            for word in sentence.words:
                if word.base_span == cur_tlink.b_text.base_span[j]:
                    phrase.append(word)
        if len(phrase) > 0:
            phrase_matches.append(phrase)
    return phrase_matches
    
# gets timex phrases' main word
def get_sentence_phrase_main_word(phrase, text_obj, current_word):
    current_word_span = text_obj.words.get(current_word)
    if current_word_span in phrase:
        return current_word_span
    else:
        for child in current_word.children:
            #print(text_obj.words.get(current_word))
            #print(f"current word: {current_word}, child: {current_word.children[i]}")
            current_word_span = get_sentence_phrase_main_word(phrase, text_obj, child)
            if current_word_span in phrase:
                return current_word_span

def get_event_timex_tlinks(tempfact_text_list):
    """
    Leiab ja tagastab sündmuste-ajaväljendite tlink labelid.
    """            
    tlinks = []
    
    # ajafaktide korpus
    for text_idx, text in enumerate(tempfact_text_list):
        # leiame juba ette ära kõik teksti ajaväljendifraaside peasõnad
        all_timexes_main_words = []
        for sentence in text.sentences:
            sent_phrases = get_sentence_phrases(text, sentence, "tlinks")
            #lausefraase_kokku+=len(sent_phrases)
            sentence_root = None
            for j in range(len(sentence.stanza_syntax)):
                if sentence.stanza_syntax.deprel[j] == 'root':
                    sentence_root = sentence.stanza_syntax[j]        
            sent_timexes_main_words = []
            for phrase in sent_phrases:
                if len(phrase) == 1:
                    sent_timexes_main_words.append(phrase[0])
                else:
                    sent_timexes_main_words.append(get_sentence_phrase_main_word(phrase, text, sentence_root))
            assert len(sent_timexes_main_words) == len(sent_phrases), "Different number of timexes and main words"
            all_timexes_main_words += sent_timexes_main_words
            
        # sündmusfraasi ja ajaväljendifraasi peasõna vektorid
        for idx1, tlink in enumerate(text["tlinks"]):
            ev_word_spans = [text.words.get(span) for span in tlink.a_text.base_span]
            tm_word_spans = [text.words.get(span) for span in tlink.b_text.base_span]
            # jätame entiteedid välja
            for span in ev_word_spans:
                for ent in text.entities:
                    if span in ent:
                        ev_word_spans = [None]
            # Üksikutel juhtudel on timexis sõnestusprobleem, jätame need välja
            if None in tm_word_spans or None in ev_word_spans:
                continue
            #sündusfraasi peasõna vektori saamiseks kasutame ära varasemalt valmistehtud peasõnade IOB-kihti
            ev_found = False
            for idx2, word in enumerate(text.gold_word_events_main):
                if word.nertag == 'B-EVENT' or word.nertag == 'I-EVENT':
                    if text.words.get(word[0]) in ev_word_spans:
                        ev_found = True
                        break
            
            if not ev_found:
                continue
            
            # võtame timexi peasõna vektori
            tm_found = False           
            for idx2, word in enumerate(text.words):
                if word in tm_word_spans and word in all_timexes_main_words:
                    tm_found = True
                    break
            
            if not tm_found:
                continue
            
            # lisame tlink labeli
            tlinks.append(tlink["rel_type"][0])

    return tlinks

In [23]:
from collections import Counter

def calculate_tlink_baseline_accuracy_score(label_list):
    most_frequent = Counter(label_list).most_common(1)[0]
    return most_frequent[0], most_frequent[1] / len(label_list)

def calculate_tlink_baseline_accuracy_score_agg(label_list):
    agg_dct = {'SIMULTANEOUS': 'OVERLAP',
              'INCLUDES': 'OVERLAP',
              'IS_INCLUDED': 'OVERLAP',
              'BEFORE': 'BEFORE',
              'AFTER': 'AFTER',
              'VAGUE': 'VAGUE'}
    label_list = [agg_dct[l] for l in label_list]
    most_frequent = Counter(label_list).most_common(1)[0]
    return most_frequent[0], most_frequent[1] / len(label_list)

In [24]:
rkogu_tlinks = get_event_timex_tlinks(rkogu_texts)
horisont_tlinks = get_event_timex_tlinks(horisont_texts)

In [25]:
rkogu_most_freq_tlink, rkogu_tlink_baseline_accuracy = calculate_tlink_baseline_accuracy_score(rkogu_tlinks)
horisont_most_freq_tlink, horisont_tlink_baseline_accuracy = calculate_tlink_baseline_accuracy_score(horisont_tlinks)

rkogu_most_freq_tlink_agg, rkogu_tlink_baseline_accuracy_agg = calculate_tlink_baseline_accuracy_score_agg(rkogu_tlinks)
horisont_most_freq_tlink_agg, horisont_tlink_baseline_accuracy_agg = calculate_tlink_baseline_accuracy_score_agg(horisont_tlinks)

In [26]:
print(f"Stenogrammides sagedaseim klass: {rkogu_most_freq_tlink}, baseline_accuracy: {rkogu_tlink_baseline_accuracy}")
print(f"Ajaloos sagedaseim klass: {horisont_most_freq_tlink}, baseline_accuracy: {horisont_tlink_baseline_accuracy}")

Stenogrammides sagedaseim klass: INCLUDES, baseline_accuracy: 0.4074074074074074
Ajaloos sagedaseim klass: IS_INCLUDED, baseline_accuracy: 0.375


In [27]:
print(f"Stenogrammides sagedaseim agreg. klass: {rkogu_most_freq_tlink_agg}, baseline_accuracy: {rkogu_tlink_baseline_accuracy_agg}")
print(f"Ajaloos sagedaseim agreg. klass: {horisont_most_freq_tlink_agg}, baseline_accuracy: {horisont_tlink_baseline_accuracy_agg}")

Stenogrammides sagedaseim agreg. klass: OVERLAP, baseline_accuracy: 0.8796296296296297
Ajaloos sagedaseim agreg. klass: OVERLAP, baseline_accuracy: 0.9519230769230769
